# RIO Radar Log Visualizer 📊
Interactive Plotly notebook for analyzing SLAM, RIO, and GPS ground-truth logs.


In [4]:
LOG_FILE = "..logs/run_20260912_140256.jsonl"

In [5]:
%pip install numpy plotly scipy


import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy.interpolate import interp1d

# ---------------------------------------------------------
# SETUP: Point this to your target .jsonl log file
# ---------------------------------------------------------
LOG_FILE = "../logs/run_20260911_181230.jsonl"



Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
  File "/tmp/ipykernel_3147/2471360254.py", line 1, in <module>
    get_ipython().run_line_magic('pip', 'install numpy plotly scipy')
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 2583, in run_line_magic
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/IPython/core/magics/packaging.py", line 105, in pip
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/ipykernel/zmqshell.py", line 795, in system_piped
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/IPython/utils/_process_posix.py", line 98, in system
ModuleNotFoundError: No module named 'pexpect'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/alok/radar/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 22

## 1. Parse Data and Align Time-Series


In [ ]:
# Parse the JSONL file
data_gps = []
data_rio = []
data_slam = []

with open(LOG_FILE, 'r') as f:
    for line in f:
        try:
            entry = json.loads(line)
            if entry.get('type') == 'gps':
                data_gps.append(entry)
            elif entry.get('type') == 'rio':
                data_rio.append(entry)
            elif entry.get('type') == 'slam':
                data_slam.append(entry)
        except json.JSONDecodeError:
            pass

print(f"Loaded {len(data_gps)} GPS, {len(data_rio)} RIO, {len(data_slam)} SLAM records.")

# Time arrays
t_gps = np.array([d['t_mono'] for d in data_gps])
t_rio = np.array([d['t_mono'] for d in data_rio])
t_slam = np.array([d['t_mono'] for d in data_slam])

# GPS Position (ENU)
if len(data_gps) > 0:
    gps_enu = np.array([d['enu'] for d in data_gps])
else:
    gps_enu = np.zeros((0,3))

# RIO Velocity and Integrated Position
if len(data_rio) > 0:
    rio_v = np.array([[d['vx'], d['vy'], d['vz']] for d in data_rio])
    # Integrate velocity to get position
    dt = np.diff(t_rio, prepend=t_rio[0])
    dt[0] = 0  # First dt is 0
    rio_pos = np.cumsum(rio_v * dt[:, np.newaxis], axis=0)
    rio_inliers = np.array([d['inliers'] for d in data_rio])
else:
    rio_pos = np.zeros((0,3))
    rio_v = np.zeros((0,3))
    rio_inliers = np.zeros(0)

# SLAM Position
if len(data_slam) > 0:
    slam_pos = np.array([d['pos'] for d in data_slam])
    slam_map_pts = np.array([d['n_map'] for d in data_slam])
else:
    slam_pos = np.zeros((0,3))
    slam_map_pts = np.zeros(0)
    
# Function to align a time series to GPS time via interpolation
def align_to_gps(t_target, data_target):
    if len(t_target) < 2 or len(t_gps) < 2:
        return np.zeros((len(t_gps), data_target.shape[1] if data_target.ndim > 1 else 1))
    f = interp1d(t_target, data_target, axis=0, bounds_error=False, fill_value="extrapolate")
    return f(t_gps)

# Align SLAM to GPS for error calculation
if len(data_slam) > 0 and len(data_gps) > 0:
    slam_pos_aligned = align_to_gps(t_slam, slam_pos)



Loaded 35 GPS, 333 RIO, 97 SLAM records.


## 2. Bird's Eye View (2D XY Trajectory)
Check for scale and lateral drift here.


In [ ]:
fig = go.Figure()

if len(gps_enu) > 0:
    fig.add_trace(go.Scatter(x=gps_enu[:,0], y=gps_enu[:,1], mode='lines+markers', name='GPS (Ground Truth)', line=dict(color='blue')))
if len(rio_pos) > 0:
    fig.add_trace(go.Scatter(x=rio_pos[:,0], y=rio_pos[:,1], mode='lines', name='RIO (Dead Reckoning)', line=dict(color='green')))
if len(slam_pos) > 0:
    fig.add_trace(go.Scatter(x=slam_pos[:,0], y=slam_pos[:,1], mode='lines+markers', name='SLAM', line=dict(color='red')))

fig.update_layout(title="2D Trajectory (Bird's Eye View)",
                  xaxis_title="East / X (m)",
                  yaxis_title="North / Y (m)",
                  yaxis=dict(scaleanchor="x", scaleratio=1),
                  height=800)
fig.show()



## 3. 3D Trajectory (Interactive)
Rotate this plot to clearly see Z-drift (sinking/floating maps).


In [ ]:
fig = go.Figure()

if len(gps_enu) > 0:
    fig.add_trace(go.Scatter3d(x=gps_enu[:,0], y=gps_enu[:,1], z=gps_enu[:,2],
                               mode='lines+markers', name='GPS', marker=dict(size=3, color='blue'), line=dict(color='blue')))
if len(rio_pos) > 0:
    fig.add_trace(go.Scatter3d(x=rio_pos[:,0], y=rio_pos[:,1], z=rio_pos[:,2],
                               mode='lines', name='RIO', line=dict(color='green')))
if len(slam_pos) > 0:
    fig.add_trace(go.Scatter3d(x=slam_pos[:,0], y=slam_pos[:,1], z=slam_pos[:,2],
                               mode='lines+markers', name='SLAM', marker=dict(size=3, color='red'), line=dict(color='red')))

fig.update_layout(title="3D Trajectory", scene=dict(
                    xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)',
                    aspectmode='data'), height=800)
fig.show()



## 4. Altitude (Z) vs Time
Isolating the Z-axis drift.


In [ ]:
fig = go.Figure()

if len(gps_enu) > 0:
    fig.add_trace(go.Scatter(x=t_gps - t_gps[0], y=gps_enu[:,2], mode='lines', name='GPS Z', line=dict(color='blue')))
if len(rio_pos) > 0:
    fig.add_trace(go.Scatter(x=t_rio - t_rio[0], y=rio_pos[:,2], mode='lines', name='RIO Z', line=dict(color='green')))
if len(slam_pos) > 0:
    fig.add_trace(go.Scatter(x=t_slam - t_slam[0], y=slam_pos[:,2], mode='lines', name='SLAM Z', line=dict(color='red')))

fig.update_layout(title="Altitude (Z) Drift over Time",
                  xaxis_title="Time (s)", yaxis_title="Z (m)")
fig.show()



## 5. Velocity Check (RIO vs GPS)
Verifying if the raw Doppler velocity from the radar is accurate.


In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    subplot_titles=("Velocity X", "Velocity Y", "Velocity Z"))

if len(gps_enu) > 1:
    dt_gps = np.diff(t_gps)
    dt_gps[dt_gps == 0] = 0.001
    gps_v = np.diff(gps_enu, axis=0) / dt_gps[:, np.newaxis]
    t_gps_mid = t_gps[:-1] + dt_gps/2
    
    fig.add_trace(go.Scatter(x=t_gps_mid - t_gps[0], y=gps_v[:,0], name='GPS Vx', line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t_gps_mid - t_gps[0], y=gps_v[:,1], name='GPS Vy', line=dict(color='blue')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t_gps_mid - t_gps[0], y=gps_v[:,2], name='GPS Vz', line=dict(color='blue')), row=3, col=1)

if len(rio_v) > 0:
    fig.add_trace(go.Scatter(x=t_rio - t_rio[0], y=rio_v[:,0], name='RIO Vx', line=dict(color='green')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t_rio - t_rio[0], y=rio_v[:,1], name='RIO Vy', line=dict(color='green')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t_rio - t_rio[0], y=rio_v[:,2], name='RIO Vz', line=dict(color='green')), row=3, col=1)

fig.update_layout(title="Velocity Comparison: RIO Doppler vs GPS Derivative", height=800)
fig.show()



## 6. Constraints / Health Metrics
Look for drops in SLAM Map Points (GICP starvation) or RIO Inliers (Doppler failures).


In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

if len(rio_inliers) > 0:
    fig.add_trace(go.Scatter(x=t_rio - t_rio[0], y=rio_inliers, name="RIO Inliers", 
                             line=dict(color='green'), fill='tozeroy'), secondary_y=False)
if len(slam_map_pts) > 0:
    fig.add_trace(go.Scatter(x=t_slam - t_slam[0], y=slam_map_pts, name="SLAM Map Points", 
                             line=dict(color='red')), secondary_y=True)

fig.update_layout(title="System Health: Feature Constraints over Time", xaxis_title="Time (s)")
fig.update_yaxes(title_text="RIO Inlier Count", secondary_y=False)
fig.update_yaxes(title_text="SLAM Map Points", secondary_y=True)
fig.show()



## 7. Absolute Error Magnitude vs Time


In [ ]:
if len(data_slam) > 0 and len(data_gps) > 0:
    # We must align the SLAM trajectory origin to the GPS origin for this to be valid.
    # Note: If there's an initial yaw offset, this error calculation will be artificially high.
    # A true analysis script (like analyze_run.py) uses Umeyama alignment to factor out yaw.
    
    # Calculate offset at start
    offset = gps_enu[0] - slam_pos_aligned[0]
    slam_pos_shifted = slam_pos_aligned + offset
    
    err = np.linalg.norm(slam_pos_shifted - gps_enu, axis=1)
    err_2d = np.linalg.norm(slam_pos_shifted[:,:2] - gps_enu[:,:2], axis=1)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_gps - t_gps[0], y=err, mode='lines', fill='tozeroy', name='3D Error', line=dict(color='red')))
    fig.add_trace(go.Scatter(x=t_gps - t_gps[0], y=err_2d, mode='lines', name='2D (XY) Error', line=dict(color='orange')))
    
    fig.update_layout(title="Absolute Drift Error vs GPS (Origin-Aligned)",
                      xaxis_title="Time (s)", yaxis_title="Error (m)")
    fig.show()
else:
    print("Need both GPS and SLAM data to compute error.")

